# deka — Reproducible pipeline: raw data → metadata → dashboard

This single notebook is the **entire data-preparation pipeline** for the Bank XYZ
branch customer-experience dashboard. Run it top-to-bottom and it regenerates the
three metadata artifacts the app consumes:

| Output | Role |
|---|---|
| `metadata/metadata.csv` | the *semantic layer* — all 632 variables tagged with Touchpoint / Construct / Dashboard Role |
| `metadata/metadata_dashboard.csv` | **the file `dashboard.py` reads** — one row per dashboard variable, with `section / touchpoint / subgroup / role / scale_type / pair_key / include` |
| `metadata/metadata_dashboard.xlsx` | hand-editable control surface for the same rows (toggle `include`, rename `subgroup`/`label`) |

After running this notebook, `streamlit run dashboard.py` works with **nothing else** —
the reproducible end state is just **raw data + this notebook + `dashboard.py`**.

**How to reproduce**

```bash
uv run --with pandas,openpyxl,nbformat,nbclient,ipykernel \
    jupyter nbconvert --to notebook --execute --inplace pipeline.ipynb
uv run --with pandas,plotly,streamlit,openpyxl streamlit run dashboard.py
```

Dependencies are declared in `pyproject.toml` / `requirements.txt`; this notebook needs
only `pandas` and `openpyxl`, and installs nothing itself.

## What this revision changed, and why

The survey asks **every branch attribute twice** — once on an *importance* scale
(`1 SANGAT TIDAK PENTING … 6 SANGAT PENTING`) and once on a *satisfaction* scale
(`1 SANGAT TIDAK PUAS … 6 SANGAT PUAS`). The two batteries carry near-identical question
text, differing only by a `" - XYZ"` suffix on the satisfaction side.

The previous revision separated the two only for Brand Image (`T_C1A` vs `T_C1B`).
Everywhere else both batteries were tagged `Attribute Satisfaction` and averaged together,
so **145 of the 314 "satisfaction" attributes were importance ratings** — which fed
importance items into the satisfaction score, the improvement-priority ranking and the
per-touchpoint radar. Section 2.3 demonstrates the problem from the raw answer labels;
section 3 fixes it by deriving each battery's scale from its own anchors instead of from
its variable prefix.

Because every touchpoint now has a correctly paired importance/satisfaction set, the
metadata also carries a **`pair_key`**, so the dashboard can run Importance-Performance
Analysis on all eight touchpoints rather than only Brand Image.

## 1 · Setup & load the raw survey

The raw file is **semicolon-delimited** with a two-line header:

- **row 0** = full Indonesian question text (used as `question` / chart labels)
- **row 1** = short variable codes (e.g. `E1A`, `T_AT3_11`) — the real column names

So data is read with `header=1` (codes become columns) while row 0 is read separately as
the question dictionary. 1,730 respondents × 632 variables.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd

# Paths resolve off the project root (run this notebook from the repo root).
ROOT = Path.cwd()
DATA = ROOT / "data" / "Deka_project_dataset_BankXYZ.csv"
# Output dir is overridable for scratch/testing; defaults to the real metadata/ folder.
OUT_DIR = Path(os.environ.get("DEKA_METADATA_OUT", ROOT / "metadata"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("pandas   :", pd.__version__)
print("ROOT     :", ROOT)
print("DATA     :", DATA, "| exists:", DATA.exists())
print("OUT_DIR  :", OUT_DIR)

pandas   : 3.0.3
ROOT     : /home/ali1618/projects/scratch
DATA     : /home/ali1618/projects/scratch/data/Deka_project_dataset_BankXYZ.csv | exists: True
OUT_DIR  : /home/ali1618/projects/scratch/metadata


In [2]:
# Full respondent table: codes as columns (header=1), values as raw strings.
df = pd.read_csv(DATA, sep=";", header=1, low_memory=False, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
print(f"shape: {df.shape[0]:,} respondents x {df.shape[1]} variables")
df.iloc[:3, :6]

shape: 1,730 respondents x 632 variables


,SERIAL,PROV,KABKOTA,CABANG,S1,S2_1
0,197513878,DKI Jakarta,Jakarta Selatan,Jakarta Selatan 1,Pria,26
1,197518398,DKI Jakarta,Jakarta Timur,Jakarta Timur 1,Pria,43
2,197544291,DKI Jakarta,Jakarta Timur,Jakarta Timur 1,Pria,30


## 2 · EDA — structure, messiness, and the *why* behind each preprocessing choice

Four things about the raw data each drive a downstream decision:

1. **Two-row header** → we need a variable→question dictionary.
2. **Ratings are stored as coded strings** (`"6  SANGAT PUAS"`, `"99 TIDAK RELEVAN"`) → we
   need a robust numeric extractor that drops N/A codes.
3. **~42% of cells are empty *by design*** (conditional routing, a competitor-only subset,
   and `99/999` "not applicable" codes) → we must **never impute**; missingness is signal.
4. **Every attribute was asked twice, on two different scales** (§2.3) → importance and
   satisfaction must be tagged apart, and can then be paired for IPA.

In [3]:
# --- the variable -> question dictionary (built from the two header rows) ---
headers_df = pd.read_csv(DATA, sep=";", header=None, nrows=2,
                         encoding="utf-8-sig", dtype=str)
long_descriptions = headers_df.iloc[0].fillna("").tolist()          # row 0: question text
short_names = headers_df.iloc[1].fillna("").str.strip().tolist()    # row 1: variable codes
data_dict = dict(zip(short_names, long_descriptions))
print(f"{len(data_dict)} variables in the data dictionary")
pd.DataFrame({"variable": list(data_dict)[:8],
              "question": [data_dict[v] for v in list(data_dict)[:8]]})

632 variables in the data dictionary


,variable,question
0,SERIAL,Serial ID
1,PROV,Provinsi
2,KABKOTA,Kab / Kota
3,CABANG,NAMA KANTOR CABANG
4,S1,Jenis kelamin
5,S2_1,Berapa tahun usia Anda? __ tahun
6,S2_2,Range Usia
7,S3,Apakah Bapak/Ibu merupakan nasabah (pemilik re...


### 2.1 · Ratings are coded strings — `clean_ratings`

Satisfaction items look like `"6  SANGAT PUAS"` / `"1 SANGAT TIDAK PUAS"`. "Not applicable"
is coded `99` / `999`, and routed-out questions are blank. The extractor pulls the leading
integer **only when it precedes ALL-CAPS label text** (a real rating), and maps `99*`,
blanks and free text to `NaN`. This is the same idea the dashboard's `to_score` uses
(`to_score` additionally caps at the scale max, e.g. 6).

In [4]:
def clean_ratings(val):
    if pd.isna(val) or str(val).strip() == "":           # blank / routed-out
        return float("nan")
    val_str = str(val).strip()
    if val_str.startswith("99"):                         # 99 / 999 = Not Applicable
        return float("nan")
    m = re.match(r"^(\d+)\s+[A-Z]", val_str)             # "6  SANGAT PUAS" -> 6
    if m:
        return float(m.group(1))
    if val_str.isdigit():                                # already a bare number
        return float(val_str)
    return val                                           # city names etc. left as-is


demo = pd.Series(["6  SANGAT PUAS", "1 SANGAT TIDAK PUAS", "99 TIDAK RELEVAN",
                  "", "JAKARTA", "5"])
pd.DataFrame({"raw": demo, "clean_ratings": demo.map(clean_ratings)})

,raw,clean_ratings
0,6 SANGAT PUAS,6.0
1,1 SANGAT TIDAK PUAS,1.0
2,99 TIDAK RELEVAN,NaN
3,,NaN
4,JAKARTA,JAKARTA
5,5,5.0


### 2.2 · ~42% missing — by design, never imputed

Overall emptiness is high, but it is **structural**, not data loss:

- **Conditional routing** — a respondent who never used the ATM never sees the ATM
  attribute items, so those cells are blank for them.
- **Competitor subset** — `*B` / competitor items (e.g. `E1B`, `G1C`) are asked only of the
  546-respondent sub-sample that also uses a competitor bank.
- **`99` / `999` N/A codes** — explicit "not applicable", mapped to `NaN` above.

Because the blanks carry meaning (this respondent wasn't in scope), we compute every metric
on the **answered** rows only and never fill. This is also why, downstream, a variable is
marked `include=1` **only if it exists in the data *and* has at least one answered row**.

> **Measuring it correctly under pandas 3.** With the string dtype pandas 3 uses for this
> file, an empty CSV field arrives as the **empty string `""`, not `NaN`** — so `df.isna()`
> reports 0.0% and hides the whole story. Emptiness has to be measured as "blank *or* NA",
> which is what `blank_share` below does.

In [5]:
def blank_share(frame):
    # Share of cells that are NA *or* whitespace-only: the honest emptiness measure.
    blank = frame.apply(lambda s: s.astype("string").fillna("").str.strip().eq(""))
    return float(blank.to_numpy().mean() * 100)


print(f"df.isna() says          : {df.isna().to_numpy().mean() * 100:0.1f}%"
      "  <- misleading under pandas 3")
print(f"blank-or-NA (the truth) : {blank_share(df):0.1f}%"
      "  (routing + competitor subset + 99/999)")


def to_score(series, max_valid=6):
    s = series.astype(str).str.extract(r"^\s*(\d+)")[0]
    s = pd.to_numeric(s, errors="coerce")
    s = s.where(s <= max_valid)
    return s.astype("Float64")


# Answered base per headline KPI: XYZ items cover everyone, competitor items only the subset.
kpi_cols = [("E1A", 6), ("E1B", 6), ("F1A", 6), ("F1B", 6), ("G1A", 10), ("G1C", 10)]
pd.DataFrame([
    {"variable": v, "scale_max": sc,
     "answered": int(to_score(df[v], sc).notna().sum()),
     "blank_pct": round(100 - to_score(df[v], sc).notna().mean() * 100, 1)}
    for v, sc in kpi_cols if v in df.columns
])

df.isna() says          : 0.0%  <- misleading under pandas 3


blank-or-NA (the truth) : 41.7%  (routing + competitor subset + 99/999)


,variable,scale_max,answered,blank_pct
0,E1A,6,1730,0.0
1,E1B,6,546,68.4
2,F1A,6,1730,0.0
3,F1B,6,546,68.4
4,G1A,10,1730,0.0
5,G1C,10,546,68.4


### 2.3 · The two batteries: every attribute was asked on **two** scales

This is the bug this revision fixes. Group the touchpoint variables into families by code
prefix and read *the anchor labels the respondents actually saw*. Each family speaks exactly
one scale, and every touchpoint has one family of each kind:

- `T_KC1` asks **importance** (`SANGAT PENTING`), `T_KC2` asks **satisfaction** (`SANGAT PUAS`)
- likewise `T_AT2`/`T_AT3`, `T_CS2`/`T_CS3`, `T_TL2`/`T_TL3`, `T_SC1`/`T_SC2`,
  `T_CA1`/`T_CA2`, `T_SL1`/`T_SL2`, and `T_C1A`/`T_C1B`

The two are indistinguishable from the variable name alone — which is why prefix-based
classification merged them — but perfectly distinguishable from the answer labels. So we
derive the scale from the data and never guess.

In [6]:
TP_OF_PREFIX = {
    "T_AT": "ATM", "T_TL": "Teller", "T_CS": "Customer Service",
    "T_CA": "Customer Advisor", "T_KC": "Branch Facilities", "T_SC": "Security",
    "T_SL": "Service Electronics", "T_C1": "Brand Image",
}
# Anchor text -> scale. The respondent-facing wording is the ground truth.
ANCHOR_SCALE = [("PENTING", "IMPORTANCE"), ("PUAS", "SATISFACTION"),
                ("SETUJU", "AGREEMENT"), ("SESUAI", "AGREEMENT"),
                ("REKOMENDASIKAN", "NPS")]


def family_of(var):
    # 'T_KC2_47' -> 'T_KC2';  'T_C1A_1' -> 'T_C1A';  'E1A' -> None
    m = re.match(r"^(T_[A-Z]+\d+[A-Z]?)", str(var))
    return m.group(1) if m else None


def touchpoint_of(var):
    for prefix, tp in TP_OF_PREFIX.items():
        if str(var).startswith(prefix):
            return tp
    return None


def anchor_labels(col):
    # The labelled (non-numeric) answers a column actually contains.
    return sorted({str(x).strip() for x in df[col].dropna().unique()
                   if re.search(r"[A-Za-z]", str(x))})


def observed_scale(col):
    # Scale implied by the anchor labels present in a column's own answers.
    seen = " | ".join(anchor_labels(col)).upper()
    for needle, scale in ANCHOR_SCALE:
        if needle in seen:
            return scale
    return None


# One scale per family, by majority vote over its labelled columns.
FAMILY_COLS = {}
for _c in df.columns:
    _f = family_of(_c)
    if _f:
        FAMILY_COLS.setdefault(_f, []).append(_c)

FAMILY_SCALE = {}
for _f, _cols in FAMILY_COLS.items():
    _votes = [s for s in (observed_scale(c) for c in _cols) if s]
    FAMILY_SCALE[_f] = pd.Series(_votes).mode()[0] if _votes else "UNLABELLED"

pd.DataFrame([
    {"family": f, "touchpoint": touchpoint_of(cols[0]), "n_cols": len(cols),
     "scale_type": FAMILY_SCALE[f],
     "example_anchor": next(iter(anchor_labels(cols[0])), "")}
    for f, cols in sorted(FAMILY_COLS.items())
])

,family,touchpoint,n_cols,scale_type,example_anchor
0,T_AT2,ATM,18,IMPORTANCE,1 SANGAT TIDAK PENTING
1,T_AT3,ATM,38,SATISFACTION,6 SANGAT PUAS
2,T_C1A,Brand Image,24,IMPORTANCE,1 SANGAT TIDAK PENTING
3,T_C1B,Brand Image,48,SATISFACTION,6 SANGAT PUAS
4,T_CA1,Customer Advisor,19,IMPORTANCE,6 SANGAT PENTING
5,T_CA2,Customer Advisor,20,SATISFACTION,6 SANGAT PUAS
6,T_CS2,Customer Service,23,IMPORTANCE,6 SANGAT PENTING
7,T_CS3,Customer Service,48,SATISFACTION,6 SANGAT PUAS
8,T_H1A,NaN,30,AGREEMENT,1 SANGAT TIDAK SETUJU
9,T_I1A,NaN,32,AGREEMENT,1 SANGAT TIDAK SESUAI


In [7]:
# The same attribute, asked twice — proof that merging the two batteries was wrong.
for imp, sat in [("T_KC1_16", "T_KC2_47"), ("T_AT2_4", "T_AT3_11")]:
    for var in (imp, sat):
        print(f"{var:10s} {data_dict[var][:62]!r}")
        print(f"           anchors: {anchor_labels(var)}")
        print(f"           mean   : {to_score(df[var]).mean():.3f}")
    print()

print("Averaged together, an 'improvement priority' list ranks how UNIMPORTANT an\n"
      "attribute is next to how UNSATISFYING it is. Those are different questions.")

T_KC1_16   'Diputarkan lagu lembut yang mengalun di area banking hall'
           anchors: ['1  SANGAT TIDAK PENTING', '6  SANGAT PENTING']
           mean   : 5.415
T_KC2_47   'Diputarkan lagu lembut yang mengalun di area banking hall - XY'
           anchors: ['1  SANGAT TIDAK PUAS', '6  SANGAT PUAS', '99  TIDAK RELEVAN']
           mean   : 5.592

T_AT2_4    'Antrian di mesin ATM tidak panjang'
           anchors: ['6  SANGAT PENTING']
           mean   : 5.866
T_AT3_11   'Antrian di mesin ATM tidak panjang - XYZ'
           anchors: ['6  SANGAT PUAS', '99  TIDAK RELEVAN']
           mean   : 5.861

Averaged together, an 'improvement priority' list ranks how UNIMPORTANT an
attribute is next to how UNSATISFYING it is. Those are different questions.


## 3 · Build the semantic metadata layer

Every variable is tagged by three deterministic rule functions:

- **`assign_touchpoint`** — prefix → touchpoint (`T_AT*`→ATM, `T_TL*`→Teller, `T_CS*`→Customer
  Service, `T_CA*`→Customer Advisor, `T_KC*`→Branch Facilities, `T_SC*`→Security,
  `T_SL*`→Service Electronics, `T_C1*`→Brand Image, `E/F/G/T_H/T_I/T_J*`→Overall).
- **`assign_construct`** — what the item *measures*. Touchpoint attributes now split into
  **Attribute Importance** vs **Attribute Satisfaction** by their observed scale (§2.3)
  rather than by prefix, and `T_I*` / `T_J*` get their own Emotion / Digitalization
  constructs instead of falling into `Other`.
- **`assign_dashboard_role`** — construct → role, which the next stage maps onto dashboard pages.

The result is written to `metadata/metadata.csv`.

In [8]:
def assign_touchpoint(var):
    if var.startswith("T_AT"): return "ATM"
    elif var.startswith("T_TL"): return "Teller"
    elif var.startswith("T_CS"): return "Customer Service"
    elif var.startswith("T_CA"): return "Customer Advisor"
    elif var.startswith("T_KC"): return "Branch Facilities"
    elif var.startswith("T_SC"): return "Security"
    elif var.startswith("T_SL"): return "Service Electronics"
    elif var.startswith(("E", "F", "G", "T_H", "T_I", "T_J")): return "Overall"
    elif var.startswith("T_C1"): return "Brand Image"
    else: return None


ATTR_PREFIXES = ("T_AT", "T_TL", "T_CS", "T_CA", "T_KC", "T_SC", "T_SL", "T_C1")


def assign_construct(var, question):
    q = str(question).lower()
    if "alasan" in q or "yang perlu diperbaiki" in q: return "Open Ended"
    if re.match(r"^(S\d|P\d)", var): return "Demographic"
    elif re.match(r"^(A|B|D|AT1|TL1|CS1)", var): return "Usage"
    elif var.startswith("G1"): return "NPS"
    elif var.startswith("E1"): return "CSAT"
    elif var.startswith(("F1", "T_H")): return "Loyalty"
    elif var.startswith("T_I"): return "Emotion"
    elif var.startswith("T_J"): return "Digitalization"
    elif "penilaian keseluruhan" in q: return "Overall Touchpoint Satisfaction"
    elif var.startswith(ATTR_PREFIXES):
        # THE FIX: the importance and satisfaction batteries are indistinguishable by
        # prefix, so split them on the scale the respondent actually saw (see §2.3).
        scale = FAMILY_SCALE.get(family_of(var))
        if scale == "IMPORTANCE":
            return "Attribute Importance"
        if scale == "SATISFACTION":
            return "Attribute Satisfaction"
        return "Other"
    return "Other"


def assign_dashboard_role(construct):
    if construct == "Demographic": return "Filter"
    elif construct in ["CSAT", "NPS", "Loyalty",
                       "Overall Touchpoint Satisfaction"]: return "KPI"
    elif construct == "Attribute Satisfaction": return "Drilldown"
    elif construct == "Attribute Importance": return "Importance"
    elif construct == "Emotion": return "Emotion"
    elif construct == "Digitalization": return "Digitalization"
    elif construct == "Open Ended": return "Text Analysis"
    return "Other"


metadata = pd.DataFrame({"variable": list(data_dict.keys()),
                         "question": list(data_dict.values())})
metadata["Touchpoint"] = metadata["variable"].apply(assign_touchpoint)
metadata["Construct"] = metadata.apply(
    lambda x: assign_construct(x["variable"], x["question"]), axis=1)
metadata["Dashboard Role"] = metadata["Construct"].apply(assign_dashboard_role)

metadata.to_csv(OUT_DIR / "metadata.csv", index=False,
                encoding="utf-8", lineterminator="\r\n")
print(f"wrote {OUT_DIR / 'metadata.csv'}  ({len(metadata)} rows)\n")
print(metadata["Dashboard Role"].value_counts().to_string())

wrote /home/ali1618/projects/scratch/metadata/metadata.csv  (632 rows)

Dashboard Role
Drilldown         303
Importance        169
KPI                54
Other              41
Emotion            32
Filter             14
Text Analysis      14
Digitalization      5


In [9]:
# Sanity peek: how the semantic layer distributes across touchpoints & constructs.
display(metadata["Touchpoint"].value_counts(dropna=False).to_frame("n_variables"))
display(metadata["Construct"].value_counts().to_frame("n_variables"))
# The split is exactly balanced per touchpoint: one importance item per satisfaction item.
display(metadata[metadata["Construct"].str.startswith("Attribute")]
        .pivot_table(index="Touchpoint", columns="Construct", values="variable",
                     aggfunc="size", fill_value=0))

,n_variables
Touchpoint,
Branch Facilities,113
Overall,87
Brand Image,72
Customer Service,71
Teller,59
ATM,56
NaN,55
Security,47
Customer Advisor,39


,n_variables
Construct,
Attribute Satisfaction,303
Attribute Importance,169
Loyalty,32
Emotion,32
Other,23
Usage,18
Overall Touchpoint Satisfaction,18
Demographic,14
Open Ended,14


Construct,Attribute Importance,Attribute Satisfaction
Touchpoint,,
ATM,18,36
Branch Facilities,35,70
Brand Image,24,48
Customer Advisor,19,19
Customer Service,23,46
Security,15,30
Service Electronics,16,16
Teller,19,38


## 4 · Build the dashboard metadata (`metadata_dashboard.{csv,xlsx}`)

This stage turns the semantic layer into the table `dashboard.py` consumes.

- **`SECTION_OF_TP`** maps each touchpoint onto one of the 5 dashboard pages (Customer
  Service / Teller / Security / Customer Advisor / Service Electronics all collapse into
  **Service Experience**).
- **`classify_subgroup`** assigns a sub-category per attribute from keyword rules on the
  question text (word-boundary match so `aman` doesn't fire on `nyaman`); first matching rule
  wins, hence most-specific rules sit on top. An importance row inherits its partner's
  sub-category so the two can never drift apart.
- **`pair_key`** links each importance row to its satisfaction row (§4.2) — this is what
  makes Importance-Performance Analysis possible on every touchpoint.
- **`include = exists_in_data AND answered_rows > 0`.** Competitor *attribute* rows are kept
  for completeness but ship `include=0`; the dashboard uses competitor data only for the KPI
  comparison (`E1B`, `F1B`, `G1C`) and the emotion comparison.

> ⚠️ **Load-bearing metadata contract.** `dashboard.py` hardcodes the exact column names
> (`variable, question, label, bank, section, touchpoint, subgroup, role, scale_max, include,
> n_terisi, ada_di_data, scale_type, pair_key`) and the exact values it filters on
> (`role ∈ {Atribut, Importance, Overall, Loyalty Driver, Emotion, Digitalization, Filter*}`,
> `bank == "XYZ"`, page names like `"Brand Image"`, touchpoints like `"Customer Service"`).
> Changing any of these makes pages silently render empty.

In [10]:
# --- KPI, filter and section/ordering constants ---
KPI_VARS = [
    ("E1A", "CSAT — Kepuasan keseluruhan", "XYZ", 6),
    ("E1B", "CSAT — Kepuasan keseluruhan", "Kompetitor", 6),
    ("F1A", "Loyalty — Kesediaan terus menggunakan", "XYZ", 6),
    ("F1B", "Loyalty — Kesediaan terus menggunakan", "Kompetitor", 6),
    ("G1A", "NPS — Kemungkinan merekomendasikan (0–10)", "XYZ", 10),
    ("G1C", "NPS — Kemungkinan merekomendasikan (0–10)", "Kompetitor", 10),
]
FILTER_VARS = [
    ("PROV", "Provinsi", "Utama"), ("KABKOTA", "Kabupaten/Kota", "Utama"),
    ("CABANG", "Cabang", "Utama"), ("S2_2", "Kelompok Usia", "Utama"),
    ("S4", "Lama Menjadi Nasabah", "Utama"), ("S1", "Gender", "Tambahan"),
    ("S7", "Frekuensi Transaksi", "Tambahan"),
]
PROFIL_VARS = ["S1", "S2_2", "S4"]
SECTION_OF_TP = {
    "Brand Image": "Brand Image", "Branch Facilities": "Branch Facilities",
    "Customer Service": "Service Experience", "Teller": "Service Experience",
    "Security": "Service Experience", "Customer Advisor": "Service Experience",
    "Service Electronics": "Service Experience", "ATM": "ATM Experience",
}
SECTION_ORDER = ["Ringkasan", "Brand Image", "Branch Facilities",
                 "Service Experience", "ATM Experience", "Filter & Profil"]
TP_ORDER = list(SECTION_OF_TP.keys())

# Emotion items mix positive and negative feelings on one agreement scale, so the negative
# ones are REVERSE-CODED: a high score on "Saya merasa Kecewa" is a bad result, not a good
# one. They must never be averaged with the positive ones.
EMO_NEGATIVE = ["tidak puas", "frustasi", "kecewa", "tertekan", "tidak bahagia",
                "diabaikan", "tergesa-gesa"]

In [11]:
# --- sub-category keyword rules (most-specific first) ---
RULES_BF = [
    ("Toilet", ["toilet"]), ("Parkir", ["parkir"]),
    ("Ruang Tunggu & Antrian", ["tunggu","menunggu","duduk","antri","antrian","mengantri"]),
    ("Banking Hall & Kenyamanan", ["banking hall","interior","bersih","kebersihan","suhu","sejuk","ac","pencahayaan","nyaman","kenyamanan","ruang"]),
    ("Lokasi & Akses", ["lokasi","akses","gedung","strategis","papan nama","tampak","dijangkau","jangkau"]),
    ("Sarana Pendukung", ["formulir","slip","brosur","alat tulis","writing","mesin","atm di cabang"]),
]
RULES_BI = [
    ("Kepercayaan & Reputasi", ["aman","keamanan","percaya","kepercayaan","terpercaya","reputasi","kinerja","dihargai","mengontrol"]),
    ("Citra & Kebanggaan", ["bangga","kebanggaan","prestis","bergengsi","terkenal","terkemuka","digunakan banyak","negara","swasta terbesar"]),
    ("Kemudahan & Teknologi", ["mudah","kemudahan","mempermudah","transaksi","channel","teknologi","online","fitur","e-channel","atm","kantor cabang","call center"]),
    ("Produk & Manfaat", ["untung","menguntungkan","keuntungan","investasi","bisnis","diskon","produk","promo","lengkap","layanan"]),
]
RULES_PETUGAS = [
    ("Penampilan", ["penampilan","rapi","seragam","menarik","berwibawa"]),
    ("Sikap & Keramahan", ["ramah","keramahan","sopan","senyum","salam","sapa","menyapa","sikap","membantu","peduli","perhatian","membukakan"]),
    ("Kecepatan & Ketanggapan", ["cepat","kecepatan","waktu","antri","tanggap","sigap","segera"]),
    ("Kompetensi & Solusi", ["mampu","kemampuan","pengetahuan","menjelaskan","informasi","solusi","akurat","teliti","ketelitian","paham","memahami","mengerti","jawab","kompeten","keluhan","masalah","kebutuhan","menghitung","memproses"]),
]
RULES_SECURITY = [("Keamanan & Kesigapan", ["aman","keamanan","menjaga","jaga","sigap","mengarahkan"])] + RULES_PETUGAS
RULES_ELEKTRONIK = [
    ("Mesin & Perangkat", ["mesin","perangkat","edc","tablet","layar","e-form"]),
    ("Kemudahan Penggunaan", ["mudah","kemudahan","jelas","petunjuk","informasi"]),
    ("Keandalan", ["berfungsi","gangguan","rusak","kerusakan","cepat"]),
]
RULES_ATM = [
    ("Keamanan", ["aman","keamanan","cctv","penjaga","dijaga"]),
    ("Keandalan Mesin", ["gangguan","rusak","kerusakan","offline","kartu","macet","berfungsi","kehabisan","tertelan","error"]),
    ("Fitur & Transaksi", ["fitur","menu","pecahan","setor","tarik","transfer","pembayaran","lengkap","transaksi","uang"]),
    ("Ketersediaan & Lokasi", ["lokasi","jumlah","tersedia","ketersediaan","mudah ditemukan","dekat","banyak","mencukupi"]),
    ("Kenyamanan & Kebersihan", ["bersih","kebersihan","nyaman","kenyamanan","ruang","suhu","terang","pencahayaan"]),
]
RULES_BY_TP = {
    "Brand Image": (RULES_BI, "Citra Umum"),
    "Branch Facilities": (RULES_BF, "Fasilitas Lain"),
    "Customer Service": (RULES_PETUGAS, "Pelayanan Umum"),
    "Teller": (RULES_PETUGAS, "Pelayanan Umum"),
    "Security": (RULES_SECURITY, "Pelayanan Umum"),
    "Customer Advisor": (RULES_PETUGAS, "Pelayanan Umum"),
    "Service Electronics": (RULES_ELEKTRONIK, "Sarana Lain"),
    "ATM": (RULES_ATM, "ATM Umum"),
}


def classify_subgroup(touchpoint, question):
    rules, default = RULES_BY_TP.get(touchpoint, ([], "Lainnya"))
    q = " " + str(question).lower() + " "
    for name, keywords in rules:
        # word-boundary match so 'aman' does not fire inside 'nyaman'
        if any(re.search(r"\b" + re.escape(k), q) for k in keywords):
            return name
    return default

### 4.2 · Labels and pair keys

Two text problems to solve at once.

**Labels were truncated to 60 characters**, which collapsed 117 of 314 attributes into
ellipsised near-duplicates no chart axis could tell apart — the question text runs to 153
characters. `clean_label` now keeps the full text, stripping only the `" - XYZ"` /
`" - kompetitor"` bank marker and the `.1`/`.2` suffixes the original CSV export added when
deduplicating repeated question strings. Where that still leaves two identical labels inside
one touchpoint (the questionnaire really does ask "Pinpad dapat berfungsi dengan baik"
twice), a ` (2)` counter is appended. The dashboard wraps long labels across lines instead
of cutting them.

**`pair_key` links the two batteries.** Matching on the visible label is fragile — the
importance and satisfaction wordings differ in stray punctuation as well as in the bank
suffix — so the key is built from a hard-normalized form of the question text (lowercased,
non-alphanumerics removed), scoped to the touchpoint, plus an occurrence counter for the
genuinely repeated questions. This pairs **all 169** attributes 1:1 on every touchpoint.

In [12]:
BANK_SUFFIX = re.compile(r"\s*[-–]\s*(xyz|kompetitor)\s*$", re.I)
DEDUP_SUFFIX = re.compile(r"\.\d+$")
# Interviewer instruction glued onto the end of one question; not part of what was asked.
NOTE_SUFFIX = re.compile(r"\s*\((note|catatan)\b.*?\)\s*$", re.I | re.S)


def clean_label(text):
    # Full question text minus the bank marker, the CSV de-duplication suffix and any
    # interviewer note. Never truncated: the dashboard wraps instead of cutting.
    t = re.sub(r"\s+", " ", str(text)).strip()
    t = DEDUP_SUFFIX.sub("", t)           # ".1" added by the original export
    t = BANK_SUFFIX.sub("", t).strip()    # " - XYZ" / " - kompetitor"
    t = NOTE_SUFFIX.sub("", t).strip()
    return DEDUP_SUFFIX.sub("", t).strip()


def norm_key(text):
    # Hard-normalized question text: the pairing key's stable core.
    return re.sub(r"[^0-9a-z]+", "", clean_label(text).lower())


def detect_bank(question):
    return "Kompetitor" if "kompetitor" in str(question).lower() else "XYZ"


def scale_type_of(var, scale_max):
    # The scale a variable was answered on: from its family's anchors, KPIs by construction.
    fam = family_of(var)
    if fam and FAMILY_SCALE.get(fam, "UNLABELLED") != "UNLABELLED":
        return FAMILY_SCALE[fam]
    var = str(var)
    if var.startswith("G1"): return "NPS"
    if var.startswith("E1"): return "SATISFACTION"
    if var.startswith("F1"): return "AGREEMENT"
    return "SATISFACTION" if scale_max == 6 else "UNLABELLED"


def normalize_semantic(meta):
    meta = meta.copy()
    meta.columns = [str(c).strip() for c in meta.columns]
    for col in ("variable", "question", "Dashboard Role", "Touchpoint", "Construct"):
        if col not in meta.columns:
            meta[col] = ""
        meta[col] = meta[col].astype(str).str.strip()
    return meta

In [13]:
def build_metadata(data_path, meta_df):
    data = pd.read_csv(data_path, sep=";", header=1, low_memory=False,
                       encoding="utf-8-sig")
    data.columns = [str(c).strip() for c in data.columns]
    meta = normalize_semantic(meta_df)
    data_cols = set(data.columns)
    rows, seen = [], set()

    def n_terisi(var, scale):
        if var not in data_cols:
            return 0
        return int(to_score(data[var], scale).notna().sum())

    def add_row(var, question, section, touchpoint, subgroup, role, scale, include):
        var = str(var).strip()
        if not var or var in seen:
            return
        seen.add(var)
        ada = var in data_cols
        n = n_terisi(var, scale) if ada else 0
        rows.append({
            "variable": var,
            "question": re.sub(r"\s+", " ", str(question)).strip(),
            "label": clean_label(question),
            "bank": detect_bank(question),
            "section": section, "touchpoint": touchpoint, "subgroup": subgroup,
            "role": role, "scale_max": scale,
            "include": int(include and ada and n > 0),
            "n_terisi": n, "ada_di_data": "Ya" if ada else "TIDAK DITEMUKAN",
            "scale_type": scale_type_of(var, scale),
            "pair_key": "",
        })

    # 4a. KPI Ringkasan
    for var, label, bank, scale in KPI_VARS:
        hit = meta.loc[meta["variable"] == var, "question"]
        q = (hit.iloc[0] if len(hit) else "") or f"{label} - {bank}"
        add_row(var, q, "Ringkasan", "", "KPI Utama", "KPI", scale, True)

    # 4b. Overall Touchpoint Satisfaction (the holistic "considering everything" item)
    ov = meta[meta["Construct"].str.lower() == "overall touchpoint satisfaction"]
    for _, r in ov.iterrows():
        tp = r["Touchpoint"]
        add_row(r["variable"], r["question"], SECTION_OF_TP.get(tp, "Service Experience"),
                tp, "Overall", "Overall", 6, detect_bank(r["question"]) == "XYZ")

    # 4c. Attributes per touchpoint: satisfaction -> Atribut, importance -> Importance.
    #     Sub-categories are classified from the satisfaction wording, then inherited by the
    #     importance partner so a pair can never land in two different sub-categories.
    subgroup_by_pair = {}
    for tp in TP_ORDER:
        sat = meta[(meta["Dashboard Role"] == "Drilldown") & (meta["Touchpoint"] == tp)]
        for _, r in sat.iterrows():
            sub = classify_subgroup(tp, r["question"])
            subgroup_by_pair.setdefault(f"{tp}|{norm_key(r['question'])}", sub)
            add_row(r["variable"], r["question"], SECTION_OF_TP[tp], tp, sub, "Atribut",
                    6, detect_bank(r["question"]) == "XYZ")
    for tp in TP_ORDER:
        imp = meta[(meta["Dashboard Role"] == "Importance") & (meta["Touchpoint"] == tp)]
        for _, r in imp.iterrows():
            sub = (subgroup_by_pair.get(f"{tp}|{norm_key(r['question'])}")
                   or classify_subgroup(tp, r["question"]))
            add_row(r["variable"], r["question"], SECTION_OF_TP[tp], tp, sub,
                    "Importance", 6, True)

    # 4d. Filters & profile
    for var, label, kelompok in FILTER_VARS:
        hit = meta.loc[meta["variable"] == var, "question"]
        add_row(var, (hit.iloc[0] if len(hit) else "") or label,
                "Filter & Profil", "", kelompok, "Filter", 0, True)
    for var in PROFIL_VARS:
        for r in rows:
            if r["variable"] == var:
                r["role"] = "Filter+Profil"

    # 4e. Loyalty Drivers (T_H* XYZ items): 15 dimensions explaining *why* customers stay.
    ld = meta[meta["variable"].str.startswith("T_H")
              & ~meta["question"].str.lower().str.contains("kompetitor")]
    for _, r in ld.iterrows():
        add_row(r["variable"], r["question"], "Ringkasan", "",
                "Loyalty Driver", "Loyalty Driver", 6, True)

    # 4f. Emotion (T_I*): kept for BOTH banks — the XYZ-vs-competitor emotional gap is the
    #     widest signal in the study. Split positive / negative because the negatives are
    #     reverse-coded (a high "Saya merasa Kecewa" is a bad outcome, not a good one).
    for _, r in meta[meta["Construct"] == "Emotion"].iterrows():
        low = clean_label(r["question"]).lower()
        polarity = "Emosi Negatif" if any(k in low for k in EMO_NEGATIVE) else "Emosi Positif"
        add_row(r["variable"], r["question"], "Ringkasan", "", polarity, "Emotion", 6, True)

    # 4g. Digitalization (T_J1_* rating items only; T_J1A/T_J2 are free-text verbatims).
    dig = meta[(meta["Construct"] == "Digitalization")
               & meta["variable"].str.match(r"^T_J1_\d+$")]
    for _, r in dig.iterrows():
        add_row(r["variable"], r["question"], "Ringkasan", "",
                "Digitalisasi Cabang", "Digitalization", 6, True)

    out = pd.DataFrame(rows)

    # --- pair_key: link each importance row to its satisfaction row -------------------
    # Scope = touchpoint + bank; an occurrence counter disambiguates questions the
    # questionnaire genuinely asks more than once inside one battery. The key deliberately
    # excludes `role`, so a pair shares one key (it appears exactly twice, once per role).
    out["__norm"] = out["question"].map(norm_key)
    pairable = out["role"].isin(["Atribut", "Importance"]) & out["touchpoint"].ne("")
    occ = (out[pairable]
           .sort_values("variable",
                        key=lambda s: s.str.extract(r"(\d+)$")[0].astype(int))
           .groupby(["touchpoint", "bank", "role", "__norm"]).cumcount())
    out.loc[pairable, "pair_key"] = (
        out.loc[pairable, "touchpoint"] + "|" + out.loc[pairable, "bank"] + "|"
        + out.loc[pairable, "__norm"] + "|"
        + occ.reindex(out.index[pairable]).astype(str))

    # --- unique labels: append " (2)" where one touchpoint repeats a question ---------
    rank = out.groupby(["touchpoint", "role", "bank", "label"]).cumcount()
    out.loc[rank > 0, "label"] = (out.loc[rank > 0, "label"]
                                  + " (" + (rank[rank > 0] + 1).astype(str) + ")")

    # tidy sort: section -> touchpoint -> subgroup -> bank -> label
    out["__s"] = out["section"].map({s: i for i, s in enumerate(SECTION_ORDER)})
    out["__t"] = out["touchpoint"].map({t: i for i, t in enumerate(TP_ORDER)}).fillna(-1)
    out = (out.sort_values(["__s", "__t", "subgroup", "bank", "label"])
              .drop(columns=["__s", "__t", "__norm"]).reset_index(drop=True))
    return out


dash = build_metadata(DATA, metadata)
dash.to_csv(OUT_DIR / "metadata_dashboard.csv", index=False,
            encoding="utf-8-sig", lineterminator="\r\n")
print(f"wrote {OUT_DIR / 'metadata_dashboard.csv'}  ({len(dash)} rows)\n")
print(dash["role"].value_counts().to_string())

wrote /home/ali1618/projects/scratch/metadata/metadata_dashboard.csv  (555 rows)

role
Atribut           303
Importance        169
Emotion            32
Overall            18
Loyalty Driver     15
KPI                 6
Digitalization      5
Filter              4
Filter+Profil       3


In [14]:
# Pairing audit: every satisfaction attribute must find exactly one importance partner.
sat = dash[(dash["role"] == "Atribut") & (dash["bank"] == "XYZ")]
imp = dash[dash["role"] == "Importance"]
audit = pd.DataFrame({
    "n_satisfaction": sat.groupby("touchpoint")["pair_key"].nunique(),
    "n_importance": imp.groupby("touchpoint")["pair_key"].nunique(),
    "paired": pd.Series({
        tp: len(set(sat.loc[sat["touchpoint"] == tp, "pair_key"])
                & set(imp.loc[imp["touchpoint"] == tp, "pair_key"]))
        for tp in sorted(sat["touchpoint"].unique())}),
}).fillna(0).astype(int)
audit["unpaired"] = audit["n_satisfaction"] - audit["paired"]
display(audit)

assert audit["unpaired"].sum() == 0, "some attributes did not pair"
for role in ("Atribut", "Importance"):
    keys = dash.loc[(dash["role"] == role) & (dash["pair_key"] != ""), "pair_key"]
    assert keys.is_unique, f"pair_key collision inside role={role}"
assert not dash.duplicated(["touchpoint", "role", "bank", "label"]).any(), "duplicate labels"
print(f"all {audit['paired'].sum()} XYZ attributes paired 1:1 · labels unique")

,n_satisfaction,n_importance,paired,unpaired
ATM,18,18,18,0
Branch Facilities,35,35,35,0
Brand Image,24,24,24,0
Customer Advisor,19,19,19,0
Customer Service,23,23,23,0
Security,15,15,15,0
Service Electronics,16,16,16,0
Teller,19,19,19,0


all 169 XYZ attributes paired 1:1 · labels unique


### 4.3 · The editable Excel control surface

`write_excel` produces a styled **Metadata** sheet (frozen header, per-section shading, an
`include` 0/1 dropdown, auto-filter), a formula-driven **Ringkasan** summary, and a
**Petunjuk** (instructions) sheet. The app prefers the `.xlsx` over the `.csv` when both
exist, so editing `include`/`subgroup`/`label` here re-shapes the dashboard without touching
code. (The `.xlsx` bytes are not reproducible — openpyxl embeds timestamps — but the
Metadata-sheet *values* are.)

In [15]:
def write_excel(out, path):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation

    C_DARK = "0F4C81"
    thin = Side(style="thin", color="B7CCE3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    f_head = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    f_body = Font(name="Arial", size=10)

    wb = Workbook()
    ws = wb.active
    ws.title = "Metadata"
    cols = list(out.columns)
    ws.append(cols)
    for _, r in out.iterrows():
        ws.append([r[c] for c in cols])
    for j, c in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=j)
        cell.font = f_head
        cell.fill = PatternFill("solid", start_color=C_DARK)
        cell.alignment = Alignment(vertical="center")
    widths = {"variable": 14, "question": 64, "label": 52, "bank": 11,
              "section": 18, "touchpoint": 18, "subgroup": 24, "role": 15,
              "scale_max": 10, "include": 9, "n_terisi": 9, "ada_di_data": 16,
              "scale_type": 14, "pair_key": 30}
    for j, c in enumerate(cols, start=1):
        ws.column_dimensions[get_column_letter(j)].width = widths.get(c, 14)
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=len(cols)):
        for cell in row:
            cell.font = f_body
            cell.border = border
            if cols[cell.column - 1] in ("question", "label"):
                cell.alignment = Alignment(wrap_text=True, vertical="top")
    sec_col = cols.index("section") + 1
    prev, shade = None, False
    for i in range(2, ws.max_row + 1):
        cur = ws.cell(row=i, column=sec_col).value
        if cur != prev:
            shade, prev = not shade, cur
        if shade:
            for j in range(1, len(cols) + 1):
                ws.cell(row=i, column=j).fill = PatternFill("solid", start_color="EFF6FD")
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(cols))}{ws.max_row}"
    inc_col = get_column_letter(cols.index("include") + 1)
    dv = DataValidation(type="list", formula1='"0,1"', allow_blank=False)
    ws.add_data_validation(dv)
    dv.add(f"{inc_col}2:{inc_col}{ws.max_row}")

    # Ringkasan sheet. Column letters are derived from the frame, so adding a metadata
    # column can never silently point these COUNTIFS at the wrong column again.
    L_sec = get_column_letter(cols.index("section") + 1)
    L_role = get_column_letter(cols.index("role") + 1)
    L_inc = get_column_letter(cols.index("include") + 1)
    last = ws.max_row
    ws2 = wb.create_sheet("Ringkasan")
    ws2.append(["Section", "Atribut kepuasan (include=1)", "Atribut kepentingan",
                "Jumlah Overall", "Total Variabel"])
    for j in range(1, 6):
        c = ws2.cell(row=1, column=j)
        c.font = f_head
        c.fill = PatternFill("solid", start_color=C_DARK)
    for i, s in enumerate([s for s in SECTION_ORDER if s in set(out["section"])], start=2):
        ws2.cell(row=i, column=1, value=s).font = f_body
        for j, role in ((2, "Atribut"), (3, "Importance"), (4, "Overall")):
            cell = ws2.cell(row=i, column=j)
            cell.value = (f'=COUNTIFS(Metadata!${L_sec}$2:${L_sec}${last},$A{i},'
                          f'Metadata!${L_role}$2:${L_role}${last},"{role}",'
                          f'Metadata!${L_inc}$2:${L_inc}${last},1)')
            cell.font = f_body
        cell = ws2.cell(row=i, column=5)
        cell.value = f'=COUNTIF(Metadata!${L_sec}$2:${L_sec}${last},$A{i})'
        cell.font = f_body
    for j, w in enumerate([22, 26, 22, 16, 14], start=1):
        ws2.column_dimensions[get_column_letter(j)].width = w
    ws2.freeze_panes = "A2"

    ws3 = wb.create_sheet("Petunjuk")
    tips = [
        "CARA MEMAKAI FILE INI", "",
        "1. Kolom 'include' = 1 berarti variabel DITAMPILKAN di dashboard, 0 disembunyikan.",
        "   Ubah ke 0 untuk atribut yang tidak ingin dimunculkan agar tampilan tetap ringan.",
        "2. Kolom 'role' menentukan CARA pemakaian variabel:",
        "     Atribut        = penilaian KEPUASAN per atribut (skala 1-6 SANGAT PUAS)",
        "     Importance     = penilaian KEPENTINGAN atribut yang sama (1-6 SANGAT PENTING)",
        "     Overall        = penilaian holistik per touchpoint",
        "     KPI            = CSAT / Loyalty / NPS di halaman Ringkasan",
        "     Loyalty Driver = 15 dimensi alasan nasabah tetap setia",
        "     Emotion        = emosi saat memakai layanan cabang (positif & negatif)",
        "     Digitalization = persepsi digitalisasi layanan cabang",
        "   JANGAN menukar 'Atribut' dengan 'Importance': keduanya memakai skala berbeda,",
        "   dan dashboard akan salah menghitung bila tertukar.",
        "3. Kolom 'pair_key' menyambungkan baris Importance dengan baris Atribut pasangannya",
        "   (dipakai grafik Importance-Performance / IPA). Jangan diubah.",
        "4. Kolom 'subgroup' adalah sub-kategori (mis. Toilet, Parkir, Ruang Tunggu).",
        "   Hasil pengelompokan otomatis — silakan rapikan/ganti namanya bila kurang pas.",
        "5. Kolom 'label' adalah teks yang muncul di grafik. Boleh diedit/dipersingkat.",
        "6. Jangan mengubah kolom 'variable' (harus sama persis dengan kode kolom di data).",
        "7. Baris bank = 'Kompetitor' untuk atribut sengaja include=0; dashboard memakai data",
        "   kompetitor hanya untuk KPI (E1B, F1B, G1C) dan grafik emosi.",
        "8. 'n_terisi' = jumlah responden yang menjawab. Perhatikan atribut dengan n kecil",
        "   (mis. Customer Advisor & Sarana Elektronik, n sekitar 70) — dashboard menandainya.",
        "9. 'ada_di_data' = TIDAK DITEMUKAN artinya kode variabel tidak ada di file data.",
        "10. Setelah selesai mengedit, cukup SIMPAN file ini (Ctrl+S).",
        "    Dashboard membaca 'metadata_dashboard.xlsx' lebih dulu; bila tidak ada,",
        "    baru membaca 'metadata_dashboard.csv'.",
    ]
    for i, t in enumerate(tips, start=1):
        c = ws3.cell(row=i, column=1, value=t)
        c.font = Font(name="Arial", size=11, bold=(i == 1),
                      color=C_DARK if i == 1 else "000000")
    ws3.column_dimensions["A"].width = 110
    ws3.sheet_view.showGridLines = False
    wb.save(path)


write_excel(dash, OUT_DIR / "metadata_dashboard.xlsx")
print(f"wrote {OUT_DIR / 'metadata_dashboard.xlsx'}")

wrote /home/ali1618/projects/scratch/metadata/metadata_dashboard.xlsx


## 5 · Self-verification

The notebook documents its own correctness: it recomputes the dashboard's headline numbers
straight from the raw data, asserts the shape of the metadata contract, and shows what the
importance/satisfaction split actually changed.

Expected headline (unfiltered Ringkasan): **CSAT 5.89 / 6 · NPS 81 · 1,730 responden**.

In [16]:
# Headline KPIs, computed exactly as dashboard.py does.
csat = to_score(df["E1A"], 6).dropna().mean()
g = to_score(df["G1A"], 10).dropna()
nps = (g >= 9).mean() * 100 - (g <= 6).mean() * 100

print(f"CSAT       : {csat:.2f} / 6")
print(f"NPS        : {nps:.0f}")
print(f"Responden  : {len(df):,}")

assert f"{csat:.2f}" == "5.89", f"CSAT changed: {csat:.2f}"
assert f"{nps:.0f}" == "81", f"NPS changed: {nps:.0f}"
assert len(df) == 1730, f"Responden changed: {len(df)}"
assert len(metadata) == 632, f"semantic rows changed: {len(metadata)}"

# The metadata contract dashboard.py depends on.
EXPECTED_ROLES = {"Atribut": 303, "Importance": 169, "Emotion": 32, "Overall": 18,
                  "Loyalty Driver": 15, "KPI": 6, "Digitalization": 5,
                  "Filter": 4, "Filter+Profil": 3}
actual = dash["role"].value_counts().to_dict()
assert actual == EXPECTED_ROLES, f"role mix changed:\n  got      {actual}\n  expected {EXPECTED_ROLES}"
assert len(dash) == sum(EXPECTED_ROLES.values()), f"rows changed: {len(dash)}"

REQUIRED_COLS = ["variable", "question", "label", "bank", "section", "touchpoint",
                 "subgroup", "role", "scale_max", "include", "n_terisi",
                 "ada_di_data", "scale_type", "pair_key"]
assert list(dash.columns) == REQUIRED_COLS, f"columns changed: {list(dash.columns)}"

# No role may mix scales: 'Atribut' is satisfaction, 'Importance' is importance. Always.
by_scale = dash.groupby(["role", "scale_type"]).size()
assert set(by_scale.loc["Atribut"].index) == {"SATISFACTION"}, by_scale.loc["Atribut"]
assert set(by_scale.loc["Importance"].index) == {"IMPORTANCE"}, by_scale.loc["Importance"]

# Labels must be readable in full: no ellipsis. 181 is the real questionnaire maximum
# (a loyalty driver that enumerates every savings product); the dashboard wraps it.
assert not dash["label"].str.endswith("…").any(), "labels are still being truncated"
assert dash["label"].str.len().max() <= 190, dash["label"].str.len().max()

print(f"\nAll assertions passed. ({len(dash)} metadata rows)")

CSAT       : 5.89 / 6
NPS        : 81
Responden  : 1,730

All assertions passed. (555 metadata rows)


In [17]:
# What the split changed: 145 importance items used to sit inside the satisfaction score.
xyz = dash[(dash["bank"] == "XYZ") & (dash["include"] == 1)]
sat_vars = list(xyz.loc[xyz["role"] == "Atribut", "variable"])
imp_vars = list(xyz.loc[xyz["role"] == "Importance", "variable"])
moved = list(xyz.loc[(xyz["role"] == "Importance") & (xyz["touchpoint"] != "Brand Image"),
                     "variable"])
old_pool = sat_vars + moved      # what the previous revision called "Atribut"

print(f"importance items reclassified out of 'Atribut' : {len(moved)}")
print(f"old merged 'Atribut' pool                      : {len(old_pool)}")
print(f"satisfaction attributes now                    : {len(sat_vars)}")
print(f"importance attributes now                      : {len(imp_vars)}\n")


def pooled_mean(cols):
    return float(pd.concat([to_score(df[c]) for c in cols], ignore_index=True).mean())


print(f"old merged pool mean : {pooled_mean(old_pool):.3f}  (satisfaction + importance)")
print(f"satisfaction only    : {pooled_mean(sat_vars):.3f}")
print(f"importance only      : {pooled_mean(imp_vars):.3f}")
print("\nThe averages barely move — which is why this went unnoticed for so long. What was\n"
      "wrong was the *meaning*: half the bars on every attribute chart answered a different\n"
      "question, and the worst bar in the improvement-priority list was an importance rating.")

importance items reclassified out of 'Atribut' : 145
old merged 'Atribut' pool                      : 314
satisfaction attributes now                    : 169
importance attributes now                      : 169



old merged pool mean : 5.839  (satisfaction + importance)


satisfaction only    : 5.834


importance only      : 5.832

The averages barely move — which is why this went unnoticed for so long. What was
wrong was the *meaning*: half the bars on every attribute chart answered a different
question, and the worst bar in the improvement-priority list was an importance rating.


In [18]:
# Attributes per sub-category (n = total rows, tampil = include=1).
summary = (dash[dash["role"].isin(["Atribut", "Importance"])]
           .groupby(["section", "touchpoint", "role", "subgroup"])
           .agg(n=("variable", "size"), tampil=("include", "sum")))
print(summary.to_string())

missing = dash[dash["ada_di_data"] != "Ya"]
print("\nVariabel tidak ditemukan di data:",
      "tidak ada" if missing.empty
      else f"\n{missing[['variable', 'section']].to_string(index=False)}")

# Thin bases the dashboard must flag rather than average away.
thin = (dash[(dash["role"] == "Atribut") & (dash["include"] == 1)
             & (dash["n_terisi"] < 100)]
        .groupby("touchpoint")["n_terisi"].agg(["size", "min", "max"]))
print("\nTouchpoint dengan basis responden kecil (n < 100):")
print(thin.to_string() if len(thin) else "  tidak ada")

                                                                              n  tampil
section            touchpoint          role       subgroup                             
ATM Experience     ATM                 Atribut    ATM Umum                    8       4
                                                  Fitur & Transaksi           8       4
                                                  Keamanan                    6       3
                                                  Keandalan Mesin             8       4
                                                  Kenyamanan & Kebersihan     2       1
                                                  Ketersediaan & Lokasi       4       2
                                       Importance ATM Umum                    4       4
                                                  Fitur & Transaksi           4       4
                                                  Keamanan                    3       3
                                